# # Class 2: Basic 2D Visual Representation

本课程将带你深入了解偏基础的计算机视觉方法，重点介绍 **卷积神经网络（CNN** 的核心原理、各种经典架构（如 ResNet 等）的发展历程，以及如何使用 PyTorch 和 TensorFlow/Keras 进行实操代码开发。

---

## 第一部分：引言与背景

### 1.1 什么是卷积神经网络？

**卷积神经网络**（Convolutional Neural Network，简称 CNN 或 ConvNet）是一种专门用于处理具有网格结构数据（如图像）的深度学习模型。

**核心特点：**
- **自动学习特征**，无需手工设计（区别于传统CV算法如SIFT/HOG）
- **局部连接和权值共享**，大幅减少了模型参数
- **平移不变性**，对图像的位置变化更加鲁棒

### 1.2 为什么需要 CNN？

**传统神经网络（全连接网络）的局限：**
- **参数过多：** 比如处理一张 224×224×3 的图像，全连接层直接处理会导致高达 15 万输入节点，参数量暴增。
- **忽略空间结构：** 图像的相邻像素往往具有很强的空间相关性，展平（Flatten）丢弃了这些信息。

**CNN 的优势：**
- 天然适合图像处理，保留空间层次结构，参数效率极高。

---
## 第二部分：CNN 核心组件

### 2.1 卷积层（Convolutional Layer）

**卷积操作原理：**
卷积核（Filter/Kernel）在输入图像上滑动，计算局部区域的加权和。

**关键概念：**
1. **卷积核大小（Kernel Size）：** 常用 3×3, 5×5。越小越能捕捉局部细节。
2. **步长（Stride）：** 卷积核每次移动的像素数。
3. **填充（Padding）：** 
   - Same Padding：保持输出尺寸不变。
   - Valid Padding：不填充，输出尺寸减小。
4. **通道数（Channels）：** 输入通道如RGB为3；输出通道等于卷积核数量，决定特征图厚度。

### 2.2 激活函数（Activation Function）

深度学习需要引入非线性，**ReLU（Rectified Linear Unit）** 是最常用的激活函数：
`f(x) = max(0, x)`
优点是计算简单，并有效缓解了梯度消失的问题。

### 2.3 池化层（Pooling Layer）

对特征图进行**降维**，减少计算量并增强平移不变性。
- **最大池化（Max Pooling）：** 取区域内最大值，保留最显著特征（最常用）。
- **平均池化（Average Pooling）：** 取区域内平均值，保留背景信息。

### 2.4 全连接层（Fully Connected Layer）

通常位于网络末端，整合提取出的高维特征，并进行最终分类或回归。现代较深的架构有时会使用全局平均池化（Global Average Pooling）替代全连接层以减少计算量。

---
## 第三部分：经典 CNN 架构发展史

CNN 的发展经历了多个里程碑式的突破，网络越来越深，表达能力越来越强：

1. **LeNet-5 (1998):**
   - 早期被用于识别手写数字 (MNIST)。
   - 包含卷积、池化、全连接基本结构（共 7 层）。

2. **AlexNet (2012):**
   - 引发深度学习热潮，赢得了 ImageNet 竞赛。
   - **创新点：** 首次使用 ReLU 激活函数、Dropout 防止过拟合、并引入了 GPU 并行计算与数据增强（Data Augmentation）。

3. **VGGNet (2014):**
   - **核心思想：** 放弃大尺寸卷积核，统一使用堆叠的 3×3 小卷积核。结构极其规整，成为许多应用的基础特征提取器。

4. **GoogLeNet / Inception (2014):**
   - 并行使用不同尺度的卷积核（1x1, 3x3, 5x5），实现了多尺度特征融合，进一步降低计算量。

5. **ResNet (2015):**
   - **核心创新：残差学习（Residual Learning）**
   - 提出跳跃连接（Skip Connection），让网络学习残差 `F(x) = H(x) - x`，而不是直接学习完整的输出。
   - **意义：** 成功解决了深度网络的梯度消失/退化问题！从此可以轻松训练上百层甚至上千层的超深网络。目前 ResNet-50 也是各类CV任务的标准骨干网络（Backbone）。

6. **MobileNet (2017+):**
   - 提出深度可分离卷积（Depthwise Separable Convolution），大幅减少参数和计算量。适合移动端部署。

---
## 第四部分：用 PyTorch 实现并理解 CNN (理论演示)

下面是一个使用 **PyTorch** 框架构建基础 CNN 网络的代码范例。通常我们通过 `torch.nn.Module` 类来定义包含卷积层（`nn.Conv2d`）和池化层（`nn.MaxPool2d`）的模型。

In [ ]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # 定义特征提取部分（多组卷积+池化提取空间特征）
        self.features = nn.Sequential(
            # 第一组卷积块: (3) -> (32)
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 池化层使长宽尺寸减半
            
            # 第二组卷积块: (32) -> (64)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  
            
            # 第三组卷积块: (64) -> (128)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  
        )
        
        # 定义分类预测部分（全连接）
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), # 假设输入图像是 32x32，3次池化后为 4x4
            nn.ReLU(),
            nn.Dropout(0.5), # Dropout 防过拟合
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)   # 提取特征
        x = self.classifier(x) # 分类预测
        return x

# 实例化模型，并尝试打印模型架构结构
pytorch_model = SimpleCNN(num_classes=10)
print(pytorch_model)

---
## 第五部分：使用 TensorFlow/Keras 进行完整实战训练

在工业界，**TensorFlow/Keras** 也被广泛用于快速搭建和训练模型。
下面我们将使用 Keras，在 **CIFAR-10** 数据集上完整演示数据的加载、构建模型、训练和验证结果的步骤。

CIFAR-10 数据集包含 60,000 张 32×32 的彩色图像，共有 10 个类别（例如飞机、汽车、鸟、猫等）。

In [ ]:
# ========== 1. 导入库与加载数据 ==========
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print("正在加载 CIFAR-10 数据集...")
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
           'dog', 'frog', 'horse', 'ship', 'truck']

print(f"训练集图像：{x_train.shape}, 测试集图像：{x_test.shape}")

# ========== 2. 数据预处理 ==========
# 将像素归一化：将像素值从 0-255 缩放到 0-1 之间，这是加速收敛的关键步骤
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [ ]:
# ========== 3. 构建 CNN 模型 ==========
model = keras.Sequential([
    # 第一组卷积块 (Conv + ReLU + Conv + ReLU + MaxPool + Dropout)
    keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),
    
    # 第二组卷积块
    keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),
    
    # 全连接分类网络
    keras.layers.Flatten(),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax') # 输出层为 10 类 Softmax
])

# 打印模型结构
model.summary()

In [ ]:
# ========== 4. 编译模型并开始训练 ==========
# 使用 Adam 优化器，损失函数设为 Sparse Categorical Crossentropy
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("开始训练 (演示时设置 epochs 为 5，实际更优模型可能需要 50 以上)...")
history = model.fit(x_train, y_train, 
                    epochs=5, 
                    batch_size=64,
                    validation_split=0.1, # 划分了 10% 的训练集用于验证
                    shuffle=True)

In [ ]:
# ========== 5. 评估模型与可视化 ==========
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"测试集最终准确率：{test_acc:.4f}")

plt.figure(figsize=(12, 4))

# 准确率可视化
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='训练准确率')
plt.plot(history.history['val_accuracy'], label='验证准确率')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('训练过程 - 准确率')

# 损失值可视化
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='训练损失')
plt.plot(history.history['val_loss'], label='验证损失')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('训练过程 - 损失')

plt.tight_layout()
plt.show()

# ========== 6. 进行预测验证 ==========
predictions = model.predict(x_test[:5])
predicted_labels = np.argmax(predictions, axis=1)

print("\n前 5 张图像的测试预测结果对比：")
for i, label in enumerate(predicted_labels):
    print(f"图像 {i+1}: 预测类别 = 【{classes[label]}】, 真实类别 = 【{classes[y_test[i][0]]}】")

---
## 第六部分：深入探讨与课程总结

### 6.1 实践中的挑战与调优技巧
训练深层视觉模型并不总是一帆风顺的，我们通常需要解决以下挑战：
1. **防止过拟合：** 需要引入 Dropout、L2 正则化、或 早停法 (Early Stopping)。
2. **数据处理：** 如果收集真实数据较难或数据过少，可以采用**数据增强（Data Augmentation）**（例如随机旋转、翻转、裁剪）；或者使用在 ImageNet 等大数据集上预训练好的权重然后进行 Fine-tuning（迁移学习）。
3. **训练优化：** 针对特定的任务选用自适应优化器如 AdamW，可能还需要搭配适当的**学习率衰减策略（如Cosine Annealing）**以提高收敛精度。

### 6.2 课程回顾
1. **CNN 的四大核心基石**：卷积层提取关键局部特征，激活层赋予网络非线性表达能力，池化层实现降噪和特征降维，全连接层用于高级维度的最后推理组合。
2. **经典网络的演进**：回顾了深度学习的发展历程，掌握了“如何构造更深且更容易训练的网络（如通过 ResNet 的残差链接解决退化）”。
3. **框架实操**：我们在本教程中学习了使用 PyTorch 构建 CNN 模型的方法，并基于 TensorFlow/Keras 在真实数据集 (CIFAR-10) 上演示了**数据读取 - 建模 - 编译训练 - 评估可视化**的一整套工程化流程。

### 6.3 计算机视觉前沿趋势 (扩展学习推荐)
1. **目标分类进阶 - 目标检测和语义分割**：不仅需要知道图里面"是什么"，还要精确知道"在哪里"（深入学习 YOLO, Faster R-CNN, Mask R-CNN）。
2. **走向大统一框架 (Vision Transformer, ViT)**：如今最前沿的 CV 网络开始引入 Transformer 架构中的 Self-Attention，能够捕捉超越局部卷积的图像全局依赖。
3. **自监督与多模态模型**：例如生成对抗网络 (GANs) 和**扩散模型 (Diffusion Model - Stable Diffusion)**，以及融合图文的多模态大模型 (CLIP)。